# 02 - Data Cleaning

In this notebook we apply the remediation rules identified in `01_data_quality.ipynb`.

All transformations are delegated to `src/cleaning.py` so that the pipeline is **deterministic and reproducible**.

For each dataset we will:
1. Load the raw file.
2. Apply the cleaning function.
3. Verify the result (no duplicates, no impossible values, normalized categories).
4. Persist the cleaned file to `data/processed/`.

Why these transformations?

| Problem | Why we do what we do |
|---------|----------------------|
| Drop exact duplicates | Avoid double-counting in every aggregation |
| Title-case names + trim | Consistent lookups, cleaner reports |
| Map regional/payment/status variants | Single source of truth for grouping |
| Drop negatives & clip outliers | Revenue, profit, quantity must be non-negative and realistic |
| Multi-format date parsing | Time-series aggregations only work with `datetime64` |
| Median imputation inside groups | Preserves category-level structure (no leakage of category-level average to a category that lacks it) |

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.cleaning import (
    clean_customers, clean_products, clean_orders,
    clean_order_items, clean_returns
)

RAW = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

def before_after(raw, clean, label):
    print(f'\n=== {label} ===')
    print(f'  rows: {len(raw):,} -> {len(clean):,}')
    print(f'  duplicates (raw): {raw.duplicated().sum():,}')
    print(f'  duplicates (clean): {clean.duplicated().sum():,}')
    nulls_raw = raw.isna().sum().sum()
    nulls_clean = clean.isna().sum().sum()
    print(f'  total nulls: {nulls_raw:,} -> {nulls_clean:,}')

## 1. Clean customers

In [ ]:
raw = pd.read_csv(RAW / 'customers.csv')
clean = clean_customers(raw)
before_after(raw, clean, 'customers')
print('\nRegion values after cleaning:')
print(clean['region'].value_counts(dropna=False).to_string())
print('\nSample rows:')
display(clean.head(3))

## 2. Clean products

In [ ]:
raw = pd.read_csv(RAW / 'products.csv')
clean = clean_products(raw)
before_after(raw, clean, 'products')
print(f'\nNegative unit_cost remaining: {(clean["unit_cost"] < 0).sum():,}')
print(f'Negative unit_price remaining: {(clean["unit_price"] < 0).sum():,}')
print('\nCategories:')
print(clean['category'].value_counts().to_string())
display(clean.head(3))

## 3. Clean orders

In [ ]:
raw = pd.read_csv(RAW / 'orders.csv')
clean = clean_orders(raw)
before_after(raw, clean, 'orders')

# Verify date parsing
print(f'\nOrders with unparsed order_date: {clean["order_date"].isna().sum():,}')
print(f'Min order_date: {clean["order_date"].min()}')
print(f'Max order_date: {clean["order_date"].max()}')

print('\nStatus values:')
print(clean['status'].value_counts(dropna=False).to_string())
print('\nPayment values:')
print(clean['payment_method'].value_counts(dropna=False).to_string())

## 4. Clean order_items

In [ ]:
raw = pd.read_csv(RAW / 'order_items.csv')
clean = clean_order_items(raw)
before_after(raw, clean, 'order_items')
print(f'\nNegative quantity: {(clean["quantity"] < 0).sum():,}')
print(f'Negative unit_price: {(clean["unit_price"] < 0).sum():,}')
print(f'Discounts > 1: {(clean["discount"] > 1).sum():,}')
print(f'Quantity stats:\n{clean["quantity"].describe().round(2).to_string()}')

## 5. Clean returns

In [ ]:
raw = pd.read_csv(RAW / 'returns.csv')
clean = clean_returns(raw)
before_after(raw, clean, 'returns')
print('\nReason values:')
print(clean['reason'].value_counts(dropna=False).to_string())

## Persist cleaned files

In [ ]:
customers = clean_customers(pd.read_csv(RAW / 'customers.csv'))
products  = clean_products(pd.read_csv(RAW / 'products.csv'))
orders    = clean_orders(pd.read_csv(RAW / 'orders.csv'))
items     = clean_order_items(pd.read_csv(RAW / 'order_items.csv'))
returns   = clean_returns(pd.read_csv(RAW / 'returns.csv'))

customers.to_csv(PROC / 'customers.csv', index=False)
products.to_csv(PROC / 'products.csv', index=False)
orders.to_csv(PROC / 'orders.csv', index=False)
items.to_csv(PROC / 'order_items.csv', index=False)
returns.to_csv(PROC / 'returns.csv', index=False)

print('Saved cleaned files to', PROC)
for f in PROC.iterdir():
    print(f'  - {f.name} ({f.stat().st_size // 1024} KB)')

## Cleaning summary

| Dataset | Rows before | Rows after | Nulls handled | Duplicates removed |
|---------|------------:|-----------:|--------------:|-------------------:|
| customers | ~5,080 | ~5,017 | yes | yes |
| products  | ~320   | ~300   | yes | yes |
| orders    | ~25,150 | ~25,000 | partial | yes |
| order_items | ~70,250 | ~70,000 | partial | yes |
| returns   | ~5,630 | ~5,600 | yes | yes |

The cleaned data is now ready for transformation and modelling.